# Invocation

**Prerequisites:** `01_what_is_a_tool.ipynb`, `02_the_registry.ipynb`. From `02_agent_runtime`: `02_the_agent_loop.ipynb`, `03_retries_and_reflection.ipynb`

Notebooks 1 and 2 were about tools sitting still: how one is defined, and how a lot of them are kept. This one is about the half-second where a tool actually runs.

Chapter 1 wrote that moment in two lines. `execute_tool_call` ends with `str(result)`, and `resolve_tools` wraps the call in `except Exception as e: content = f"Error: {e}"`. Every ending is a string, and every failure is the same kind of failure.

Four things go wrong there, and none of them look like a bug in the tool:

- the tool never comes back, and the agent waits forever
- a failure that was ours gets handed to the model, which tries to work around it
- the result is far too big to put in a conversation, and it is re-sent on every turn afterwards
- a call times out, gets retried, and the email goes twice

**By the end you'll have:**
- Timed a call that hangs, and seen why chapter 2's own time budget can't stop it
- Built `invoke()`, which returns a `Result` for every ending instead of raising
- Sorted failures by who can do something about them — the model, or you
- Cut a 7,600-token result down to what a conversation can carry, while the program keeps all of it
- Marked a tool as unsafe to repeat, and watched a timeout say so
- Run it inside chapter 2's ReAct loop, still without changing `02_agent_runtime`

## Setup

In [1]:
import json
import sys
import time
from pathlib import Path
from typing import Annotated

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))
sys.path.append(str(Path.cwd().parent.parent / "02_agent_runtime"))

from chat import count_tokens
from invoke import Result, ToolError, invoke, truncate
from registry import Registry
from tools import tool

## A call that never comes back

Chapter 2's `Runtime` has a `max_seconds` budget. It is checked at the top of each pass, before the next component runs:

```python
while not self.finished:
    reason = self._stop_reason(conversation)
    if reason is not None:
        return self._end_gracefully(...)
    component = self.next()
    output = component.run(conversation)
```

So the budget is looked at *between* components, and never while one is running. `Act` calls the tool function directly. Until that call returns, nothing else in the process gets a turn — including the code whose job is to notice that time has run out. A tool waiting on a server that will never answer takes the whole agent with it.

Here is a tool that takes five seconds, called both ways:

In [2]:
@tool
def slow_lookup(city: Annotated[str, "The city to look up"]) -> str:
    """Look up a city in a slow system."""
    time.sleep(5)
    return f"finally, something about {city}"

started = time.monotonic()
slow_lookup(city="Oslo")
print(f"straight through:  {time.monotonic() - started:.1f}s")

started = time.monotonic()
result = invoke(slow_lookup, {"city": "Oslo"}, timeout=2)
print(f"through invoke():  {time.monotonic() - started:.1f}s")
print(result)

straight through:  5.0s


through invoke():  2.0s
'slow_lookup' took longer than 2 seconds and was given up on. It only reads, so calling it again is safe.


`invoke` runs the tool on a separate thread and waits on it for as long as it is allowed to:

```python
pool = concurrent.futures.ThreadPoolExecutor(max_workers=1)
try:
    return pool.submit(function, **args).result(timeout=timeout)
finally:
    pool.shutdown(wait=False)
```

Two details in those five lines, both of which cost real time to find.

**It isn't a `with` block.** Leaving one waits for the thread to finish, which is the one thing the timeout exists to avoid. Written the tidy way, the call gives up after two seconds and then sits there for the remaining three anyway.

**And it does not stop the tool.** Python cannot kill a thread from outside. `shutdown(wait=False)` stops *us* waiting; the five-second sleep is still running, still holding whatever it holds, and it will finish in its own time. So what a timeout buys is the agent's attention back, not the work stopped. Stopping it for real means running it in a separate process, which is what `09_sandbox` is for.

## Which failures belong in the conversation

Chapter 1 turned every failure into `f"Error: {e}"` and sent it back. Chapter 2 was more careful. `DebugThink` gets the schema and every failed attempt, and tries to repair the arguments. But it still treats every failure as the same kind of thing, and retries all of them three times.

They are not the same kind of thing. What separates them is not how bad they are. It is **who can do something about it**:

In [3]:
@tool
def get_weather(city: Annotated[str, "The city to look up"]) -> str:
    """Look up the current weather in a city."""
    readings = {"tokyo": 18, "oslo": 4, "cairo": 33}
    if city.lower() not in readings:
        raise ToolError(f"No weather station for {city}. Try a larger nearby city.")
    return f"{readings[city.lower()]} degrees celsius, clear skies in {city}."

@tool
def check_pressure(city: Annotated[str, "The city to look up"]) -> str:
    """Look up the air pressure in a city."""
    return {"tokyo": 1013}[city.lower()]      # a plain bug: no key for anywhere else

for tool_, args in [(get_weather, {"city": "Oslo"}),
                    (get_weather, {"town": "Oslo"}),
                    (get_weather, {"city": "Narnia"}),
                    (check_pressure, {"city": "Oslo"})]:
    r = invoke(tool_, args)
    print(f"{r.kind:14} goes back to the model: {str(r.for_the_model):5}  {str(r).splitlines()[0][:60]}")

ok             goes back to the model: True   4 degrees celsius, clear skies in Oslo.
bad_arguments  goes back to the model: True   2 validation errors for get_weather_args
refused        goes back to the model: True   No weather station for Narnia. Try a larger nearby city.
broken         goes back to the model: False  check_pressure is broken: KeyError: 'oslo'


Four endings, three of which the model can act on:

- **ok** — nothing to fix.
- **bad_arguments** — the model sent `town` instead of `city`. It caused this, and the message names the field, so it can correct it. Nothing ran, so nothing happened.
- **refused** — the tool ran and said no. There is no weather station for Narnia, and the tool suggests what might work instead. That is information, not a fault.
- **broken** — `check_pressure` has a bug. The model did nothing wrong and can do nothing about it.

The last one is the one worth being strict about. Handing "KeyError: 'oslo'" back to a model invites it to work around a problem it cannot see. It will retry, reword, try neighbouring cities — and spend the whole budget on a line of code that was never going to work. So `for_the_model` is `False`, the message is written for whoever is reading the run, and the loop is expected to stop rather than think harder.

A tool says which kind it is by what it raises. `ToolError` means "I ran, and the answer is no". Anything else means the tool is broken:

```python
except ToolError as e:
    return Result(tool.name, "refused", str(e), ...)
except Exception as e:
    return Result(tool.name, "broken", f"{tool.name} is broken: {type(e).__name__}: {e}", ...)
```

Nothing here raises. Every ending comes back as a `Result` with a `kind`, and what to do about it is left to the caller. That is the same line chapter 2 draws when it says `Act` carries things out and never decides.

## A result too big to keep

`str(result)` is fine for a weather reading. Here is a tool that returns two hundred customer records, which is not a lot of records:

In [4]:
@tool
def list_customers(region: Annotated[str, "Which region to list"]) -> str:
    """List every customer in a region, with their notes."""
    return json.dumps([{"id": i, "name": f"customer {i}", "region": region,
                        "notes": "renewed in March, asked about volume pricing "}
                       for i in range(200)])

whole = invoke(list_customers, {"region": "nordics"})
print("as it comes:", count_tokens(whole.content), "tokens")

cut = invoke(list_customers, {"region": "nordics"}, max_tokens=200)
print("with max_tokens=200:", count_tokens(cut.content), "tokens")
print()
print(str(cut)[-220:])
print()
print("and the program still has all of it:", len(json.loads(cut.value)), "records")

as it comes: 7601 tokens
with max_tokens=200: 217 tokens

ed in March, asked about volume pricing "}, {"id": 4, "name": "customer 4", "region": "nordics", "notes": "renewed in March, asked about volume pricing "}, {"id": 5, "name":
[... cut short: 7,601 tokens in all, 200 kept]

and the program still has all of it: 200 records


Notebook 2 measured what thirty tool *descriptions* cost on every turn. A result is worse, because it lands in the conversation and stays there. That is 7,601 tokens re-sent on every turn for the rest of the run — to answer a question that probably needed three records.

Two things make the cut safe to do.

**The note at the end.** Without it, the model reads a sentence that stops mid-word. It has no way to tell a short answer from a long one with the end cut off. With it, the model knows there was more and can ask a narrower question.

**`value` keeps the original.** The conversation gets what it can afford; the program keeps everything. Chapter 1's notebook 9 made the same move with `notes`, and chapter 2 with `conversation.pending`. Not everything the agent knows has to be in what the model is shown.

## A call that isn't safe to repeat

A timeout is the one ending where nobody knows what happened. The call was given up on, not cancelled — as established above, the tool is very likely still running. It may have finished a moment later. It may have already sent the email.

For a tool that only reads, calling it again costs a little time. For a tool that sends, pays, or deletes, calling it again is a second email. Nothing about the timeout tells you which one you are looking at, so the tool has to have said in advance:

In [5]:
@tool(repeatable=False)
def send_email(to: Annotated[str, "Who to send it to"],
               body: Annotated[str, "What to say"]) -> str:
    """Send an email immediately."""
    time.sleep(5)
    return f"sent to {to}"

reading = invoke(slow_lookup, {"city": "Oslo"}, timeout=1)
sending = invoke(send_email, {"to": "priya@example.com", "body": "Friday deploy"}, timeout=1)

for r in (reading, sending):
    print(f"{r.tool:12} safe to retry: {str(r.safe_to_retry):5}  {r}")

slow_lookup  safe to retry: True   'slow_lookup' took longer than 1 seconds and was given up on. It only reads, so calling it again is safe.
send_email   safe to retry: False  'send_email' took longer than 1 seconds and was given up on. It may or may not have gone through. Do not call it again -- check first.


The two messages are different on purpose, because they are read by different things. The text goes to the model, which is the one deciding what to do next. `safe_to_retry` is for the loop around it, which is the thing that might quietly try again.

This matters more than it looks, because chapter 2's `Act` retries a failed tool call three times by default. A `send_email` that times out at four seconds and actually goes through at five, retried twice, is three emails and one very confused recipient.

## Into the loop

`Registry.invokers()` wraps every tool, so chapter 2's `Act` calls `invoke` without knowing it. `Result.__str__` is its message, and `execute_tool_call` finishes with `str(result)` — so what reaches the conversation is the text, and the rest of the `Result` stays behind.

`schemas=None` is deliberate: with it, the arguments are checked once, inside `invoke`, instead of once by `Act` and again by us.

The question asks for a city the weather tool doesn't have, so the tool refuses and the model has to do something about it:

In [6]:
from chat import Conversation
from runtime import Runtime
from observe import Observe
from think import Think
from act import Act

weather = Registry([get_weather])

runtime = Runtime(
    loop=[
        Observe(event="What's the weather in Vestby, Norway?"),
        Think(tools=weather.tools, allow_fork=False),
        Act(functions=weather.invokers(timeout=10, max_tokens=500), schemas=None),
        Observe(),
    ],
    repeat_from=1,
    max_iterations=10,
)

conversation = runtime.run(Conversation())
for m in conversation.messages:
    print(f"- {m['role']:9} {m['content']!r}")

- user      "What's the weather in Vestby, Norway?"
- decision  '{"city":"Vestby, Norway"}'
- tool      'There is no weather station for Vestby, Norway, so you should try a larger nearby city.'
- decision  'There is no weather station for Vestby, Norway, so you should try a larger nearby city.'
- assistant 'There is no weather station for Vestby, Norway, so you should try a larger nearby city.'


It ran to a clean finish, which is the first thing to check. The tool refused, the refusal arrived as an ordinary tool result, and the loop carried on and answered. Nothing raised, nothing was retried three times, nothing crashed.

The interesting part is the second line, and it did not read like this the first time this notebook was run:

```
the tool said:  "No weather station for Vestby, Norway. Try a larger nearby city."
first run:      "There is no weather station for Vestby, Norway."
now:            "There is no weather station for Vestby, Norway, so you should try
                 a larger nearby city."
```

Chapter 2's `Observe` turns a raw tool result into an observation, and with no extractor set for a tool it does that with an LLM. Asked for "one plain factual sentence", it dropped the suggestion. That half looked like commentary rather than fact.

So the fix went into the prompt in `02_agent_runtime/observe.py`: keep anything the tool itself said about what to do next — a suggestion, a constraint, an alternative to try, the reason it refused. That is part of the result, not commentary on it.

The general point outlives this one tool. `invoke` writes a failure message for the model to act on, and between `invoke` and the model sits something that rewrites it. Careful wording at the tool boundary buys nothing if the next component paraphrases it away.

One thing the run still does *not* do: told to try a larger nearby city, the model passed that on to the user instead of calling `get_weather` again with Oslo. That is defensible. The tool named no particular city, so picking one would have been a guess, and nothing in this loop told it to keep trying until something worked. Getting the message through intact is what the fix was for. Deciding what to do with it is the loop's job, and that is chapter 2's territory.

There is a seam here worth naming rather than papering over. `Act` has its own retry loop, from chapter 2, which repairs arguments with `DebugThink` and tries three times before giving up. `invoke` has its own view of what is worth retrying. Right now the two don't know about each other. With `schemas=None`, bad arguments come back as an ordinary tool result instead of raising. So `Act` never sees a failure to repair, and the correction happens out in the conversation, where the next `Think` can see it.

That arrangement is defensible: a repair the conversation can see beats one hidden inside `Act`. But it was not designed. It fell out of the wiring. Chapter 2's retry policy was written before any of this existed, and reconciling the two properly is a change to `02_agent_runtime`, which this chapter has deliberately not made.

## What you built

✓ Timed a five-second tool against a two-second limit, and saw why chapter 2's `max_seconds` cannot stop it: the budget is checked between components, and a blocking call never gives the loop a turn

✓ Built `invoke()`, which returns a `Result` for every ending instead of raising — so deciding what to do next stays with the caller

✓ Found the two details that make a timeout actually work: not using a `with` block, which waits for exactly what you gave up on, and knowing that `shutdown(wait=False)` frees the agent without stopping the tool

✓ Sorted failures by who can act on them: wrong arguments and a tool that says no both go to the model, while a bug in the tool does not, because inviting a model to work around a problem it cannot see spends the whole budget on nothing

✓ Made `ToolError` the way a tool says "I ran, and the answer is no", with anything else meaning it is broken

✓ Cut a 7,601-token result down to 217 for the conversation while `value` kept all 200 records, and put a note where the cut is so the model knows there was more

✓ Marked a tool as `repeatable=False`, so a timeout says "this may or may not have gone through" instead of "safe to try again" — the one thing a timeout can never work out for itself

✓ Ran it all through chapter 2's ReAct loop with `Registry.invokers()`, and caught `Observe` summarizing the useful half out of a carefully worded refusal — which is why the prompt in `02_agent_runtime/observe.py` now says to keep whatever the tool said about what to do next

✓ Named the seam this leaves rather than hiding it: `Act`'s own three-times retry was written before `invoke` existed, and the two still don't know about each other

**Next:** `04_verification.ipynb` — checking the arguments going in is not the same as checking what came out. A tool that reports success and did nothing, and what it takes to notice.